# Submission Evaluation - Stage 2

In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [1]:
import os

os.getcwd()

'C:\\Users\\brown\\Documents\\GitHub\\march-machine-learning-mania'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

KAGGLE_DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data\kaggle")))
SUBMISSION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\submissions"))
)
DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data")))

In [19]:
# Load data
submission_df = pd.read_csv(KAGGLE_DATA_PATH / "SampleSubmissionStage2.csv")
m_seeds = pd.read_csv(KAGGLE_DATA_PATH / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(KAGGLE_DATA_PATH / "WNCAATourneySeeds.csv")
m_teams = pd.read_csv(KAGGLE_DATA_PATH / "MTeams.csv")
w_teams = pd.read_csv(KAGGLE_DATA_PATH / "WTeams.csv")
tourney_rounds = pd.read_csv(DATA_PATH / "tourney_round_lookup.csv")

# Extract Season, Team1, and Team2 from ID
submission_df[["Season", "Team1", "Team2"]] = submission_df["ID"].str.split(
    "_", expand=True
)
submission_df[["Season", "Team1", "Team2"]] = submission_df[
    ["Season", "Team1", "Team2"]
].astype(int)

# Get Gender
submission_df["Gender"] = np.where(submission_df["Team1"] >= 3000, "Women", "Men")

# Merge Team Name
mw_teams = pd.concat([m_teams, w_teams], ignore_index=True)
submission_df = submission_df.merge(
    mw_teams[["TeamID", "TeamName"]], left_on=["Team1"], right_on=["TeamID"], how="left"
).rename(columns={"TeamName": "TeamName1"})
submission_df = submission_df.merge(
    mw_teams[["TeamID", "TeamName"]], left_on=["Team2"], right_on=["TeamID"], how="left"
).rename(columns={"TeamName": "TeamName2"})
submission_df.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)


# Function to get seed values
def get_seed_values(seed):
    return int(seed[1:3])


m_seeds["SeedValue"] = m_seeds["Seed"].apply(get_seed_values)
w_seeds["SeedValue"] = w_seeds["Seed"].apply(get_seed_values)

mw_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)


# Merge Seeds
submission_df = submission_df.merge(
    mw_seeds, left_on=["Season", "Team1"], right_on=["Season", "TeamID"], how="left"
).rename(columns={"Seed": "Seed1", "SeedValue": "SeedValue1"})
submission_df = submission_df.merge(
    mw_seeds, left_on=["Season", "Team2"], right_on=["Season", "TeamID"], how="left"
).rename(columns={"Seed": "Seed2", "SeedValue": "SeedValue2"})
submission_df.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)


# Ensure Seed1 and Seed2 are strings
submission_df["Seed1"] = submission_df["Seed1"].astype(str)
submission_df["Seed2"] = submission_df["Seed2"].astype(str)

# Create StrongSeed and WeakSeed
submission_df["StrongSeed"] = submission_df[["Seed1", "Seed2"]].min(axis=1)
submission_df["WeakSeed"] = submission_df[["Seed1", "Seed2"]].max(axis=1)


# Function to clean seed values
def clean_seed(seed):
    return seed[:-1] if seed[-1] in ["a", "b"] else seed


submission_df["StrongSeed"] = submission_df["StrongSeed"].apply(clean_seed)
submission_df["WeakSeed"] = submission_df["WeakSeed"].apply(clean_seed)

submission_df["StrongSeedValue"] = submission_df[["SeedValue1", "SeedValue2"]].min(
    axis=1
)
submission_df["WeakSeedValue"] = submission_df[["SeedValue1", "SeedValue2"]].max(axis=1)


# Merge Round Info
submission_df = submission_df.merge(
    tourney_rounds,
    left_on=["StrongSeed", "WeakSeed"],
    right_on=["StrongSeed", "WeakSeed"],
    how="left",
)

In [20]:
submission_df

,ID,Pred,Season,Team1,Team2,Gender,TeamName1,TeamName2,Seed1,SeedValue1,Seed2,SeedValue2,StrongSeed,WeakSeed,StrongSeedValue,WeakSeedValue,Round,Slot
0,2025_1101_1102,0.5,2025,1101,1102,Men,Abilene Chr,Air Force,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN
1,2025_1101_1103,0.5,2025,1101,1103,Men,Abilene Chr,Akron,nan,NaN,W13,13.0,W13,nan,13.0,13.0,NaN,NaN
2,2025_1101_1104,0.5,2025,1101,1104,Men,Abilene Chr,Alabama,nan,NaN,W02,2.0,W02,nan,2.0,2.0,NaN,NaN
3,2025_1101_1105,0.5,2025,1101,1105,Men,Abilene Chr,Alabama A&M,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN
4,2025_1101_1106,0.5,2025,1101,1106,Men,Abilene Chr,Alabama St,nan,NaN,Y16a,16.0,Y16,nan,16.0,16.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131402,2025_3477_3479,0.5,2025,3477,3479,Women,East Texas A&M,Mercyhurst,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN
131403,2025_3477_3480,0.5,2025,3477,3480,Women,East Texas A&M,West Georgia,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN
131404,2025_3478_3479,0.5,2025,3478,3479,Women,Le Moyne,Mercyhurst,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN
131405,2025_3478_3480,0.5,2025,3478,3480,Women,Le Moyne,West Georgia,nan,NaN,nan,NaN,nan,nan,NaN,NaN,NaN,NaN


In [21]:
submission_df[(submission_df["Gender"] == "Men") & (submission_df["Round"] == 1)]

,ID,Pred,Season,Team1,Team2,Gender,TeamName1,TeamName2,Seed1,SeedValue1,Seed2,SeedValue2,StrongSeed,WeakSeed,StrongSeedValue,WeakSeedValue,Round,Slot
732,2025_1103_1112,0.5,2025,1103,1112,Men,Akron,Arizona,W13,13.0,W04,4.0,W04,W13,4.0,13.0,1.0,R1W4
1322,2025_1104_1352,0.5,2025,1104,1352,Men,Alabama,Robert Morris,W02,2.0,W15,15.0,W02,W15,2.0,15.0,1.0,R1W2
1816,2025_1106_1120,0.5,2025,1106,1120,Men,Alabama St,Auburn,Y16a,16.0,Y01,1.0,Y01,Y16,1.0,16.0,1.0,R1Y1
2941,2025_1110_1181,0.5,2025,1110,1181,Men,American Univ,Duke,W16a,16.0,W01,1.0,W01,W16,1.0,16.0,1.0,R1W1
5109,2025_1116_1242,0.5,2025,1116,1242,Men,Arkansas,Kansas,Z10,10.0,Z07,7.0,Z07,Z10,7.0,10.0,1.0,R1Z7
6287,2025_1120_1384,0.5,2025,1120,1384,Men,Auburn,St Francis PA,Y01,1.0,Y16b,16.0,Y01,Y16,1.0,16.0,1.0,R1Y1
7220,2025_1124_1280,0.5,2025,1124,1280,Men,Baylor,Mississippi St,W09,9.0,W08,8.0,W08,W09,8.0,9.0,1.0,R1W8
10592,2025_1136_1277,0.5,2025,1136,1277,Men,Bryant,Michigan St,Y15,15.0,Y02,2.0,Y02,Y15,2.0,15.0,1.0,R1Y2
12064,2025_1140_1433,0.5,2025,1140,1433,Men,BYU,VCU,W06,6.0,W11,11.0,W06,W11,6.0,11.0,1.0,R1W6
16408,2025_1155_1270,0.5,2025,1155,1270,Men,Clemson,McNeese St,X05,5.0,X12,12.0,X05,X12,5.0,12.0,1.0,R1X5


In [22]:
true_results_df = submission_df[submission_df["Round"] >= 1].copy()

In [24]:
true_results_df[
    [
        "ID",
        "Gender",
        "Seed1",
        "Team1",
        "TeamName1",
        "Seed2",
        "Team2",
        "TeamName2",
        "Round",
        "Slot",
    ]
].to_csv(DATA_PATH / "true_results_df.csv", index=False)